## **Evaluación del RAG mediante RAGAS**

En este notebook examinamos el RAG que hemos definido anteriormente mediante el sistema RAGAS, que es el actual standard de la industria. Mostraremos los resultados que, de todas formas, solo nos dejan parcialmente satisfechos, pues no obtenemos muchos datos de las preguntas/respuestas del Nivel 3.

Como Judge hemos utilizado el LLM v1 de DeepSeek. Inicialmente probamos utilizar el propio Gemma2, pero la RAM explotaba, y siempre tuve dudas de lo correcto que sería examinarse un modelo a sí mismo.

Hemos utiizado solo 10 ejemplos. Reconocemos que se trata de un número pequeño, pero, pese a ello, sí que creemos que obtenemos conclusiones relevantes.

Comenzamos cargando Unsloth, y el resto de librerías necesarias.

In [ ]:
!pip install unsloth sentence-transformers faiss-cpu ragas langchain datasets langchain-huggingface

Instalamos OpenAI, para poder interaccionar con el LLM v1 de DeepSeek.

In [ ]:
!pip install openai

Ya que tenemos modelos y datasets subidos a Hugging Face descargamos las librerías necesarias.

In [ ]:
# !pip install langchain-huggingface
from datasets import Dataset
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

Ahora cargamos los embeddings y los metadata. Tuvimos serios problemas con el hecho de que los metadata tuvieran formato de un dataset de pandas.

Finalmente, pudimos crear el vector vector_db, pero fue bastante complicado.

In [ ]:
import faiss
import pickle
import pandas as pd
from pathlib import Path
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document # Updated import path

# Custom implementation of InMemoryDocstore if it's no longer directly importable
# This class provides the '_dict' attribute expected by FAISS for its docstore.
class InMemoryDocstore:
    def __init__(self):
        self._dict = {}

    def search(self, _id: str) -> Document:
        return self._dict.get(_id)

# 1. Cargamos el RAG que ya tienes
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

folder_path = "/content/drive/MyDrive/Practica_Bootcamp_IA_4/"
faiss_index_filepath = Path(folder_path) / "electoral_index_2011-19.faiss"
pickle_filepath = Path(folder_path) / "metadata_rows_2011-19.pkl"

# Load FAISS index
faiss_index = faiss.read_index(str(faiss_index_filepath))

# Load the DataFrame from the pickle file
with open(pickle_filepath, "rb") as f:
    df_metadata = pickle.load(f)

# Prepare docstore and index_to_docstore_id
docstore = InMemoryDocstore()
index_to_docstore_id = []

for i, row in df_metadata.iterrows():
    doc_id = str(i) # Use row index as document ID
    # Assuming 'metadata_vectorial' column contains the document content
    doc = Document(page_content=row['metadata_vectorial'], metadata={})
    docstore._dict[doc_id] = doc
    index_to_docstore_id.append(doc_id)

# Create FAISS instance
vector_db = FAISS(
    embedding_function=embeddings,
    index=faiss_index,
    docstore=docstore,
    index_to_docstore_id=index_to_docstore_id,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Ahora cargamos la librería Ragas, concretamente evaluate() y las métricas.

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipython-input-9873/3120045098.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
/tmp/ipython-input-9873/3120045098.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collect

Ahora cargamos las 10 preguntas, con las correspondientes respuestas esperadas, que consideramos como Ground Truth.

In [ ]:
import pandas as pd
from datasets import Dataset

In [ ]:
import pandas as pd
df_eval = pd.read_json('/content/drive/MyDrive/Practica_Bootcamp_IA_4/text_ragas_10.jsonl', lines=True)

Empezamos a rellenar las listas que utilizará la librería Ragas.

In [ ]:
questions = df_eval['input_text'].tolist()
ground_truths = df_eval['target_text'].tolist()
answers = []
contexts = []

Ahora cargamos nuestro modelo con Unsloth.

In [ ]:
from unsloth import FastLanguageModel

/tmp/ipython-input-9873/2814113929.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
model_id = "GuillermoBarrio/modelo-Gemma_2-9b-electoral-5-elecciones"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id, # <--- Aquí pones el ID de Hugging Face
    max_seq_length = 4096, # Increased max_seq_length to accommodate longer inputs
    load_in_4bit = True
)

Unsloth: If you want to finetune Gemma 2, install flash-attn to make it faster!
To install flash-attn, do the below:

pip install --no-deps --upgrade "flash-attn>=2.6.3"
==((====))==  Unsloth 2026.2.1: Fast Gemma2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Ponemos el modelo en inferencia.

In [ ]:
FastLanguageModel.for_inference(model)

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 3584, padding_idx=0)
    (layers): ModuleList(
      (0-41): 42 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=3584, out_features=2048, bias=False)
          (v_proj): Linear4bit(in_features=3584, out_features=2048, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=3584, bias=False)
          (rotary_emb): GemmaFixedRotaryEmbedding()
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=3584, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm):

Ahora preparamos la función que obtenga las respuestas del nuestro modelo a las 10 preguntas.

In [ ]:
def tu_funcion_de_generacion(pregunta, contextos):
    contexto_unido = "\n\n".join(contextos)

    # IMPORTANTE: Usamos el formato exacto del entrenamiento
    prompt = f"### Instrucción:\n{pregunta}\n\n### Contexto:\n{contexto_unido}\n\n### Respuesta:"

    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens = 512,
        use_cache = True,
        temperature = 0.1,
        repetition_penalty = 1.1
    )

    respuesta_completa = tokenizer.batch_decode(outputs)[0]

    # Separamos para obtener solo lo que el modelo escribió después de "### Respuesta:"
    if "### Respuesta:" in respuesta_completa:
        respuesta_solo = respuesta_completa.split("### Respuesta:")[1].strip()
    else:
        respuesta_solo = respuesta_completa.strip()

    return respuesta_solo.replace(tokenizer.eos_token, "")

Ahora ejecutamos la función anterior, y almacenamos las respuestas dadas por nuestro modelo.

In [ ]:
import torch

n = 0

#answers = []
#contexts = []

for q in questions:

    n += 1

    print(n)
    # a) FAISS busca los fragmentos (contextos)
    # Buscamos k=3 o el número que prefieras según la complejidad [cite: 3, 26, 59]
    docs = vector_db.similarity_search(q, k=3)
    c_list = [doc.page_content for doc in docs]
    contexts.append(c_list)

    # b) El modelo genera la respuesta usando la función anterior
    res = tu_funcion_de_generacion(q, c_list)
    answers.append(res)

    # c) LIMPIEZA CRÍTICA DE VRAM
    torch.cuda.empty_cache()

1
2
3
4
5
6
7
8
9
10


Ahora definimos a nuestro Judge, que será el LLM v1 de DeepSeek. Lógicamente, no mostramos nuestra key.

In [ ]:
from ragas.llms import llm_factory
from ragas.embeddings import LangchainEmbeddingsWrapper
from openai import OpenAI
from google.colab import userdata

# DeepSeek API Configuration
# DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
DEEPSEEK_API_KEY = 'Mi_clave_DeepSeek'
DEEPSEEK_API_BASE = 'https://api.deepseek.com/v1'
DEEPSEEK_MODEL_NAME = 'deepseek-chat' # Or 'deepseek-v2' if that's the desired model

# Create an OpenAI client instance pointing to DeepSeek API
openai_client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_API_BASE,
    # model_kwargs={"extra_body": {"response_format": {"type": "json_object"}}}
    # max_tokens=4096,
    # temperature=0
)

# Initialize ragas_llm using llm_factory with the DeepSeek model and client
ragas_llm = llm_factory(DEEPSEEK_MODEL_NAME, client=openai_client, max_tokens=4096, temperature=0)


# Initialize ragas_embeddings using your existing embeddings
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

print("Ragas evaluators configured to use DeepSeek API.")

Ragas evaluators configured to use DeepSeek API.


/tmp/ipython-input-9873/3549460591.py:26: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)


Ahora creamos el diccionario con el formato Ragas.

In [ ]:
from datasets import Dataset

# Creamos el diccionario con la estructura que RAGAS exige
# Es vital que los nombres de las columnas sean exactamente estos
data_dict = {
    "question": questions,      # Las preguntas de tu .jsonl
    "answer": answers,          # Las respuestas generadas por tu modelo Gemma-2
    "contexts": contexts,       # Los fragmentos recuperados de FAISS (lista de listas)
    "ground_truth": ground_truths # Las respuestas correctas de tu .jsonl
}

# Convertimos el diccionario a un objeto Dataset de Hugging Face
dataset_ragas = Dataset.from_dict(data_dict)

print("Dataset de RAGAS preparado con éxito.")

Dataset de RAGAS preparado con éxito.


In [ ]:
ragas_llm

InstructorLLM(provider='openai', model='deepseek-chat', client=<Instructor:sync>, temperature=0, max_tokens=4096, top_p=0.1)

Finalmente, ejecutamos el Ragas. Tuvimos que hacer cambios, porque mandaba demasiadas tareas a la vez.

Lo cierto es que para solo 10 preguntas, o un total de 10*4 = 40 pasos, tardó casi una hora. Eso fue lo que nos decidió a tomar los resultados de solo 10 preguntas.

En algunos casos avanzaba muy rápido, en otros se quedaba fijo durante 10 minutos.

Encontramos problemas con la evaluación de las preguntas largas, Nivel 3, que gemini interpretó como que el v1 se volvía demasiado quisquilloso. En fin.

In [ ]:
from ragas import evaluate, RunConfig
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    answer_correctness,
    context_recall
)
import torch
import os

# 1. Configuramos el RunConfig para limitar hilos y reintentos
# Esto sustituye al antiguo is_async=False y ayuda a no saturar DeepSeek
run_config = RunConfig(
    max_workers=1,       # Ejecuta una evaluación a la vez (evita el 0/40 bloqueado)
    timeout=60,          # Da tiempo suficiente para respuestas largas
    max_retries=3        # Si falla por red, reintenta automáticamente
)

# 2. Aseguramos las métricas
metrics = [faithfulness, answer_relevancy, answer_correctness, context_recall]

# 3. Limpieza de memoria
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache()

# 4. EJECUTAR EVALUACIÓN
results = evaluate(
    dataset=dataset_ragas,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=run_config  # <--- Pasamos la configuración aquí
)

# 5. Resultados
df_results = results.to_pandas()
print("¡Evaluación completada con éxito!")
display(df_results.head())

/tmp/ipython-input-9873/3801875416.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipython-input-9873/3801875416.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipython-input-9873/3801875416.py:2: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import (
/tmp/ipython-input-9873/3801875416.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated a

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[8]: InstructorRetryException(<failed_attempts>

<generation number="1">
<exception>
    The output is incomplete due to a max_tokens length limit.
</exception>
<completion>
    ChatCompletion(id='f487e86b-76d0-4bad-8f5e-331d6b32a98a', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n    "statements": [\n        {\n            "statement": "The analysis examines the relationship between average personal income and voting behavior in municipalities of Madrid province.",\n            "reason": "The context provides electoral and socioeconomic data for specific sections in Móstoles and Madrid, but does not explicitly state that an analysis is being conducted or that relationships are being examined.",\n            "verdict": 0\n        },\n        {\n            "statement": "The analysis focuses on vote polarization between the center-right bloc (PP+Cs) and the left bloc (PSOE+UP).",\n   

¡Evaluación completada con éxito!


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,answer_correctness,context_recall
0,### Instrucción:\n¿Qué partido ganó las elecci...,[elecciones: Noviembre 2019. municipio: Entram...,### Contexto:\nelecciones: Noviembre 2019. mun...,"El las elecciones de Diciembre 2015, el ganado...",0.714286,0.681106,0.622007,0.0
1,### Instrucción:\n¿Cuántos votos consiguió el ...,[elecciones: Junio 2016. municipio: Talarn. pr...,### Contexto:\nelecciones: Junio 2016. municip...,El PP en la sección 3802803005 en las eleccion...,0.857143,0.644206,0.438813,0.0
2,### Instrucción:\nConsiderando los datos de lo...,[elecciones: Abril 2019. municipio: Móstoles. ...,### Contexto:\nelecciones: Abril 2019. municip...,"Para abordar esta pregunta compleja, realizare...",NaN,0.527913,NaN,0.0
3,### Instrucción:\n¿Cuál es el resultado de la ...,[elecciones: Noviembre 2019. municipio: Fígols...,### Contexto:\nelecciones: Noviembre 2019. mun...,El resultado de la columna % IU de la sección ...,0.950000,0.660835,NaN,0.0
4,### Instrucción:\n¿Cuál fue el % PSOE mínimo e...,[elecciones: Junio 2016. municipio: Campos. pr...,### Contexto:\nelecciones: Junio 2016. municip...,El % PSOE mínimo en la provincia de Islas Bale...,0.888889,0.569793,0.447622,0.0


Vamos, de todas formas, con los resultados. Damos más detalles en la memoria, pero queda claro que nuestro modelo está bien entrenado, porque tiene una faithfulness de casi el 0.9, independientemente del nivel de la pregunta, incluso el nivel 3.

Donde falla algo, o bastante, es el answer correctness, que es debido a que el RAG no selecciona bien las filas. Por ejemplo, le podemos pedir datos sobre un municipio en una elección, y nos da datos de otro municipio en otra elección.

Con todo, esto creemos que lo mitigamos en la propia app, haciendo un limpiado de filas entre el faiss y en reranker. El resultado es entonces mejor.

In [ ]:
# Añadimos los metadatos de tu JSONL al resultado para agrupar
df_results['nivel'] = df_eval['metadata'].apply(lambda x: x.get('nivel_original', 'N/A'))

# Agrupamos para ver dónde brilla y dónde falla el RAG
analisis_tfm = df_results.groupby('nivel')[['faithfulness', 'answer_correctness']].mean()
print(analisis_tfm)

        faithfulness  answer_correctness
nivel                                   
nivel1      0.875162            0.372727
nivel2      0.894444            0.447622
nivel3      0.894737                 NaN


El fichero de resultados está disponible en el repositorio del trabajo final, dentro de la carpeta Datasets.

In [ ]:
df_results.to_csv("resultados_evaluacion_ragas_tfm_10.csv", index=False)